# Datamine CORREL Process Example

This notebook demonstrates how to configure and run the **`correl`** process wrapper in `dmstudio`.

## Process Description

## CORREL

Calculates bivariate statistics on pairs of numeric fields in a file. All results are output as the bottom left-hand triangle of a full two-dimensional matrix. Output is written to the display, and includes the following:-

* a summary of the mean and variance of each numeric field.

* sample number matrix.

* correlation matrix.

* covariance matrix.

* F ratio matrix.

* paired t test matrix.

The sample number matrix contains the number of pairs of samples for each pair of fields. This shows when one or other of the fields has absent data values. If none of the records contain missing data then all the elements of the matrix will contain the same value.

The mean and variance of each field shown in the summary statistics are calculated using all the available data, whereas the matrices are calculated using only data where neither sample is missing. Thus if there is missing data it is not possible for example to calculate the correlation coefficient from the covariance and variances. Bivariate statistics are calculated by default for all pairs of numeric fields. Thus if for example the data file contains sample co-ordinates statistics will be calculated for the X and Y co-ordinates.

Also see this online [Knowledge Base article](<https://datamine.freshdesk.com/en/support/solutions/articles/19000084719-correl-calculation-of-bivariate-statistics-for-numerical-fields>) (Internet connection required).

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

* **fieldlst** (Undefined):
  File to supply selected fields.
  Required=No

### Output Files:

### Fields:

* **fields** (Undefined : Undefined):
  First field to be correlated. No fields specified means all.
  Default=Undefined
  Required=No

* **fieldnam** (Character : FIELDLST):
  Field in FIELDLST to identify selected fields.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('correl')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute correl
print("Running correl...")
dm_cmd.correl(
    in_i='t_assays',  # required
    # fieldlst_i='optional',  # optional
    # fields_f=['AU'],  # optional
    # fieldnam_f='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("correl execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_correl_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")